In [1]:
import sys
import pandas as pd
import numpy as np
import scanpy as sc

from sklearn.cluster import KMeans
import scipy.stats
from scipy.stats import hypergeom
from sklearn.metrics import pairwise_distances
from itertools import combinations


import seaborn as sns
import matplotlib.pyplot as plt
plt.rcParams.update({'axes.labelsize' : 'large'}) 
from matplotlib.backends.backend_pdf import PdfPages

import sys
import os

from tqdm import tqdm
from tqdm.contrib.concurrent import process_map

import multiprocessing as mp

import gc
import warnings
import time
import pickle

from sklearn.metrics import pairwise_distances
import torch

from importlib import reload
import util_functions

/project/GCRB/Hon_lab/s223695/anaconda3/envs/scanpy_gpu/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
!nvidia-smi

Wed May 29 12:42:19 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla V100-PCIE-32GB           Off | 00000000:3B:00.0 Off |                    0 |
| N/A   27C    P0              26W / 250W |    161MiB / 32768MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [3]:
!free -m

              total        used        free      shared  buff/cache   available
Mem:         385502       16669      299028         393       69804      367056
Swap:        131071        1490      129581


<h3>Load PCA matrix and gRNA-Cell name dictionary</h3>

In [4]:
reload(util_functions)
input_file = "/project/shared/gcrb_igvf/data/shared/TF_Perturbseq_full/H5AD/V0.1_TF_Perturbseq_full_sgRNACells_filtered_w_embedding_full_dataset.h5ad"
sgRNA_file = "/project/GCRB/Hon_lab/s215194/Single_Cell/TF_perturbseq_full/_all_lanes_combined/aggr_dataframe/aggr_combined_df_full.pkl"
dict_file = "./gRNA_dict.pickle"
pca_file = "./pca_dataframe.pickle"
non_target_file = "./non_target_cell_name.pickle"

obsm_key = "X_pca"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

(X,gRNA_dict) = util_functions.load_files(input_file,sgRNA_file,dict_file,pca_file,obsm_key)

read input
read pickle
read from dictionary


In [5]:
gRNA_transcript = {}

for key in gRNA_dict.keys():
    transcript_name = util_functions.extract_transcript_name(key)
    if len(gRNA_dict[key]) > 20:
        if transcript_name in gRNA_transcript.keys():
            gRNA_transcript[transcript_name] += [key]
        else:
            gRNA_transcript[transcript_name] = [key]

<h3>Calculate e-distance between gRNA per target</h3>

In [ ]:
#compute every pair in sgRNA make rank-distance plot
total_permute = 1000
non_target_pick = 2000
batch_num_basic = 120
result_df_dict = {}

transcript_list = np.array(list(gRNA_transcript.keys()))
transcript_list = transcript_list[transcript_list!="non-targeting"]

pbar = tqdm(transcript_list)

for target in pbar:
    total_combis_tmp = []
    
    gRNA_list_target = np.array(gRNA_transcript[target])
    total_gRNA_num = len(gRNA_list_target)
    
    for num_count in range(1,total_gRNA_num):
        for combis_tmp in combinations(gRNA_list_target,num_count):
            other_gRNA = gRNA_list_target[~np.isin(gRNA_list_target,combis_tmp)]
            for num_other_count in range(1,total_gRNA_num):
                for other_tmp in combinations(other_gRNA,num_other_count):
                    total_combis_tmp += [(list(combis_tmp),list(other_tmp))]

    total_combis = util_functions.get_unique_list(total_combis_tmp)

    res = []
    for index_combi,(combi_test1,combi_test2) in enumerate(total_combis):
        pbar.set_postfix({"processing combi":index_combi+1,
                          "total combi":len(total_combis)
                         })
        cell_test1 = np.unique(np.concatenate([gRNA_dict[name] for name in combi_test1]))
        cell_test2 = np.unique(np.concatenate([gRNA_dict[name] for name in combi_test2]))
        total_cell_num = len(cell_test1)+len(cell_test2)
        if  total_cell_num > 1500:
             batch_num = batch_num_basic //2
        elif total_cell_num > 3000:
            batch_num = batch_num_basic //4
        else :
            batch_num = batch_num_basic
            
        obs_edist = util_functions.permutation_test(X,cell_test1,cell_test2,device,
                                                    batch_num,total_permute,return_permute=False)
        res += [obs_edist.item()]

    result_df = pd.DataFrame(zip(res,total_combis),columns=["e_dist","combis"])
    result_df = result_df.sort_values(["e_dist"]).reset_index(drop=True)
    result_df["rank"] =result_df.index.tolist()

    for gRNA_name_tmp in gRNA_list_target:
        result_df[gRNA_name_tmp] = result_df["combis"].apply(lambda x: (gRNA_name_tmp in x[0]) or (gRNA_name_tmp in x[1]))
    result_df_dict[target] = result_df

 94%|█████████▎| 2046/2183 [09:32<00:24,  5.60it/s, processing combi=288, total combi=301]

In [9]:
p_val_dict = {}

test_three_gRNA = False
disco_val_dict = {}

total_permute = 1000
batch_num_basic = 120

for target in tqdm(result_df_dict.keys()):
    result_df = result_df_dict[target]
    
    total_gRNA_list = gRNA_transcript[target]
    total_cell_list = [gRNA_dict[i] for i in total_gRNA_list]
    total_cell_num = len(np.concatenate(total_cell_list))

    if  total_cell_num > 1500:
         batch_num = batch_num_basic //2
    elif total_cell_num > 3000:
        batch_num = batch_num_basic //4
    else :
        batch_num = batch_num_basic
            
    obs_fvalue, fvalue_list = util_functions.disco_test(X,total_cell_list,device,batch_num=batch_num)
    disco_pvalue = (fvalue_list>obs_fvalue).sum().item()/total_permute
    disco_val_dict[target] = disco_pvalue
    
    if len(total_gRNA_list)==1:
        for gRNA_name_tmp in gRNA_transcript[target]:
            p_val_dict[gRNA_name_tmp] = 0.0
        continue
        
    if disco_pvalue > 0.05 or len(total_gRNA_list)==2:
        for gRNA_name_tmp in gRNA_transcript[target]:
            p_val_dict[gRNA_name_tmp] = 1.0
        continue
    
    else:
        num_sig_diff = result_df.shape[0] // 2
        pval_seq_tmp = []
        result_df_right = result_df.iloc[(result_df.shape[0]-num_sig_diff):,:]
        
        for gRNA_name_tmp in gRNA_transcript[target]:   
            [M, n, N, x] = [result_df.shape[0],np.sum(result_df[gRNA_name_tmp]),num_sig_diff,np.sum(result_df_right[gRNA_name_tmp])]
            prb = hypergeom.cdf(x, M, n, N)
            p_val_dict[gRNA_name_tmp] = np.round(1-prb,4)
            pval_seq_tmp.append(p_val_dict[gRNA_name_tmp])
        # print(pval_seq_tmp)
        # print(calculate_q(pval_seq_tmp))
outlier_df = pd.Series(p_val_dict)
outlier_df.name = "pval_outlier"

dist_cut_off = 0
p_val_cutoff = 0.001

outlier_df.to_csv("sgRNA_outlier_dist_v3.csv")

100%|██████████| 2183/2183 [17:18<00:00,  2.10it/s]


In [10]:
!nvidia-smi

Wed May 29 13:09:57 2024       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.129.03             Driver Version: 535.129.03   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  Tesla V100-PCIE-32GB           Off | 00000000:3B:00.0 Off |                    0 |
| N/A   31C    P0              36W / 250W |  26259MiB / 32768MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [11]:
cols = 2

with PdfPages("gRNA_outlier_dist_v3.pdf") as pdf:
    for target in tqdm(result_df_dict.keys()):
        gRNA_num = len(gRNA_transcript[target])
        if gRNA_num == 1:
            continue
        
        rows = gRNA_num//cols + 1
        fig, ax = plt.subplots(rows,cols, figsize=(16,4*rows))
        
        result_df = result_df_dict[target]
        
        for index, gRNA_name_tmp in enumerate(gRNA_transcript[target]):
            col_index = index % cols
            row_index = index // cols
            sns.scatterplot(data=result_df,x="rank",y="e_dist",hue=gRNA_name_tmp,ax=ax[row_index,col_index])
            ax[row_index,col_index].text(0,(result_df["e_dist"].max()+result_df["e_dist"].min())/2,
                                         outlier_df[gRNA_name_tmp])
        pdf.savefig()
        plt.close()

100%|██████████| 2183/2183 [11:23<00:00,  3.20it/s]


<h3>Filtering out bad gRNA using Bonferroni correction</h3>

In [18]:
is_significant_list = []
for index,row in outlier_df.items():
    transcript_name = util_functions.extract_transcript_name(index)
    num_gRNA = len(gRNA_transcript[transcript_name])
    
    if row > 0.05*num_gRNA:
        is_significant_list.append(True)
    else:
        is_significant_list.append(False)

outlier_df_clear = outlier_df[is_significant_list]

In [19]:
outlier_df_clear.to_csv("sgRNA_outlier_dist_clear_v3.csv")